### 這個colab檔案負責處理詩集的斷詞

In [ ]:
!pip install -U ckip-transformers

In [ ]:
import gdown
import os
import glob
import re
import pandas as pd
import shutil # For copying files

# ==========================================
# 1. 下載 GitHub 資料夾中的所有 .txt 檔案
# ==========================================
folder_url = 'https://github.com/Joanne334/Intro-to-AI-g8/tree/5e3e2e59e87941219f09850b63bd3c8c6dc46ded/txt_datas/poems/*'

# Extract repository information from the folder_url
match = re.match(r'(https://github.com/[^/]+/[^/]+)/tree/[^/]+/(.*)/\*', folder_url)
if not match:
    raise ValueError("Invalid GitHub folder URL format provided.")

repo_base_url = match.group(1)
source_folder_relative_path = match.group(2)
repo_name = repo_base_url.split('/')[-1] # e.g., Intro-to-AI-g8
repo_url = repo_base_url + '.git'

# Define output folder name
output_dir = 'downloaded_files'

# Check and create output folder
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已創建輸出資料夾: {output_dir}")
else:
    # Clear existing content in output_dir to ensure fresh download
    for item in os.listdir(output_dir):
        item_path = os.path.join(output_dir, item)
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
    print(f"已清空現有輸出資料夾: {output_dir}")


# Clean up existing cloned repository if it exists from previous runs
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print(f"已移除舊的克隆儲存庫資料夾: {repo_name}")


# Clone the GitHub repository
print(f"正在從 {repo_url} 克隆儲存庫...")
!git clone "{repo_url}"
print(f"儲存庫 {repo_name} 克隆完成。")

# Define the source path for the .txt files within the cloned repository
source_folder_in_repo = os.path.join(repo_name, source_folder_relative_path)

# Check if the source folder exists after cloning
if os.path.exists(source_folder_in_repo):
    print(f"正在從 '{source_folder_in_repo}' 複製 .txt 檔案到 '{output_dir}'...")
    # Get all .txt files from the source folder
    txt_files_to_copy = glob.glob(os.path.join(source_folder_in_repo, "*.txt"))

    if not txt_files_to_copy:
        print(f"警告：在 {source_folder_in_repo} 中找不到任何 .txt 檔案。")
    else:
        for file_path in txt_files_to_copy:
            shutil.copy(file_path, output_dir)
            print(f"已複製檔案: {os.path.basename(file_path)}")
        print(f"所有 .txt 檔案已複製到: {output_dir}")
else:
    print(f"錯誤：在克隆的儲存庫中找不到資料夾 '{source_folder_in_repo}'。請檢查 '{source_folder_relative_path}' 路徑是否正確。")

# Clean up the cloned repository folder to avoid clutter
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print(f"已移除克隆的儲存庫資料夾: {repo_name}")

print(f"\n資料夾內容已下載至: {output_dir}")

已創建輸出資料夾: downloaded_files
正在從 https://github.com/Joanne334/Intro-to-AI-g8.git 克隆儲存庫...
Cloning into 'Intro-to-AI-g8'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 146 (delta 9), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 107.18 KiB | 5.64 MiB/s, done.
Resolving deltas: 100% (9/9), done.
儲存庫 Intro-to-AI-g8 克隆完成。
正在從 'Intro-to-AI-g8/txt_datas/poems' 複製 .txt 檔案到 'downloaded_files'...
已複製檔案: 【楊喚詩集】犁.txt
已複製檔案: 【楊喚詩集】童詩：森林的詩.txt
已複製檔案: 【楊喚詩集】鑰匙.txt
已複製檔案: 【楊喚詩集】小樓.txt
已複製檔案: 【楊喚詩集】童詩：小蟋蟀.txt
已複製檔案: 【楊喚詩集】二十四歲.txt
已複製檔案: 【楊喚詩集】失眠夜.txt
已複製檔案: 【楊喚詩集】童詩：美麗島.txt
已複製檔案: 【楊喚詩集】童詩：毛毛是個好孩子.txt
已複製檔案: 【楊喚詩集】童詩：小紙船.txt
已複製檔案: 【楊喚詩集】詩簡.txt
已複製檔案: 【楊喚詩集】感謝  ──致安徒生  .txt
已複製檔案: 【楊喚詩集】童詩：水果們的晚會.txt
已複製檔案: 【楊喚詩集】懷劉妍.txt
已複製檔案: 【楊喚詩集】醒來.txt
已複製檔案: 【楊喚詩集】載重.txt
已複製檔案: 【楊喚詩集】童詩：七彩的虹.txt
已複製檔案: 【楊喚詩集】鄉愁.txt
已複製檔案: 【楊喚詩集】八月的斷想.txt
已複製檔案: 【楊喚詩集】雨.txt
已複製檔案: 【楊喚詩集】花與果實.

In [ ]:
# ==========================================
# 2. 定義切片與資料夾處理邏輯
# ==========================================
def process_all_txt_in_folder(directory_path, max_length=200):
    all_segments_data = []

    # 遞迴尋找資料夾內所有的 .txt 檔案
    txt_files = glob.glob(os.path.join(directory_path, "**/*.txt"), recursive=True)

    if not txt_files:
        print("警告：在指定的資料夾中找不到任何 .txt 檔案。")
        return pd.DataFrame()

    print(f"找到 {len(txt_files)} 個 txt 檔案，開始進行切片處理...")

    for file_path in txt_files:
        # 讀取單一文本
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except UnicodeDecodeError:
            print(f"警告：檔案 {file_path} 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。")
            try:
                with open(file_path, 'r', encoding='big5') as f:
                    lines = f.readlines()
            except UnicodeDecodeError as e:
                print(f"錯誤：檔案 {file_path} 無法以 BIG5 編碼讀取，跳過此檔案。錯誤訊息: {e}")
                continue # Skip this file if it can't be decoded with BIG5 either
        except Exception as e:
            print(f"讀取檔案 {file_path} 時發生未知錯誤，跳過此檔案。錯誤訊息: {e}")
            continue

        if not lines:
            continue

        # ==========================================
        # === 詩歌體專用切分邏輯 (取代原本的換行切分) ===
        # ==========================================

        # 1. 移除每行多餘空白，並用換行符號連接，將整首詩視為一個完整字串
        content = "\n".join([line.strip() for line in lines if line.strip()])

        segments = []

        # 2. 判斷長度：若整首詩未超過限制，直接整首包成單一片段
        if len(content) <= max_length:
            segments.append(content)
        else:
            # 3. 若超過限制，利用正規表達式進行軟切分
            # 考量到詩歌體常缺乏句號，這裡加入逗號「，」與換行「\n」作為輔助斷句點
            sentences = re.split(r'([。！？；，\n])', content)
            current_segment = ""

            for i in range(0, len(sentences) - 1, 2):
                sentence = sentences[i]
                punctuation = sentences[i+1] if (i+1) < len(sentences) else ""
                full_sentence = sentence + punctuation

                # 檢查當前片段加上新句子是否會超過上限
                if len(current_segment) + len(full_sentence) <= max_length:
                    current_segment += full_sentence
                else:
                    # 若超載，把當前收集到的片段存起來 (並去除頭尾多餘空白/換行)
                    if current_segment.strip():
                        segments.append(current_segment.strip())
                    current_segment = full_sentence

            # 存下最後剩餘的句子
            if current_segment.strip():
                segments.append(current_segment.strip())


        # 將該檔案切出來的所有片段，加入總清單中
        for i, seg in enumerate(segments):
            all_segments_data.append({
                "source_file": os.path.basename(file_path), # 紀錄來源檔名以便追蹤
                "segment_id": i + 1,
                "text_segment": seg,
                "length": len(seg)
            })
    # 將所有資料轉換為 DataFrame
    return pd.DataFrame(all_segments_data)


# ==========================================
# 3. 執行處理並匯出結果
# ==========================================
# 執行批次處理 (設定 max_length=200)
df_all_segments = process_all_txt_in_folder(output_dir, max_length=200)

if not df_all_segments.empty:
    print(f"\n處理完成！共產出 {len(df_all_segments)} 個文本片段。")

    # 匯出成 CSV 檔案供後續 BERT 訓練使用
    csv_filename = "processed_segments.csv"
    df_all_segments.to_csv(csv_filename, index=False, encoding='utf-8-sig') # 使用 utf-8-sig 避免 Excel 開啟亂碼
    print(f"已將結果儲存為 {csv_filename}")

    # 顯示前 5 筆結果預覽
    display(df_all_segments.head())

找到 39 個 txt 檔案，開始進行切片處理...
警告：檔案 downloaded_files/【楊喚詩集】犁.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
錯誤：檔案 downloaded_files/【楊喚詩集】犁.txt 無法以 BIG5 編碼讀取，跳過此檔案。錯誤訊息: 'big5' codec can't decode byte 0xff in position 173: illegal multibyte sequence
警告：檔案 downloaded_files/【楊喚詩集】童詩：森林的詩.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】鑰匙.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】小樓.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】童詩：小蟋蟀.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】二十四歲.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】失眠夜.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】童詩：美麗島.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】童詩：毛毛是個好孩子.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】童詩：小紙船.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
警告：檔案 downloaded_files/【楊喚詩集】詩簡.txt 無法以 UTF-8 編碼讀取，嘗試使用 BIG5 編碼。
錯誤：檔案 downloaded_files/【楊喚詩集】詩簡.txt 無法以 BIG5 編碼讀取，跳過此檔案。錯誤訊息: 'big5' codec can't decode by

,source_file,segment_id,text_segment,length
0,【楊喚詩集】童詩：森林的詩.txt,1,【楊喚詩集】童詩：森林的詩\n楊喚\n「太陽好！\n早晨好！」\n喜鵲小姐第一個睜開眼睛，\...,188
1,【楊喚詩集】童詩：森林的詩.txt,2,就被請走給老杉樹公公去看病。\n不帶體溫計，\n也沒有聽診器，\n他仔細地給老杉樹檢查，\n...,200
2,【楊喚詩集】童詩：森林的詩.txt,3,就飛東飛西去訪問，\n讓辛苦了一天的朋友們坐下來休息，\n讓她唱幾支這世界上最好聽的歌。\n...,195
3,【楊喚詩集】童詩：森林的詩.txt,4,不停地對著月亮和星星講故事，\n一歡喜起來就怪聲怪氣地笑。,28
4,【楊喚詩集】鑰匙.txt,1,【楊喚詩集】鑰匙\n楊喚\n我有一串鑰匙，\n那拙笨短小的就像白癡和侏儒，\n那姣好玲瓏的一...,124


In [ ]:
# 將 df_all_segments 中 'text_segment' 欄位的換行符號 '\n' 替換為空格
# 這樣可以將多行文本合併成單行，同時保留原本由換行符號分隔的詞語之間的空間
df_all_segments['text_segment'] = df_all_segments['text_segment'].str.replace('\n', ' ', regex=False)

print("已將 'text_segment' 欄位中的換行符號替換為空格。")

# 顯示修改後的數據前幾行以供檢視
display(df_all_segments[['text_segment']].head())

已將 'text_segment' 欄位中的換行符號替換為空格。


,text_segment
0,【楊喚詩集】童詩：森林的詩 楊喚 「太陽好！ 早晨好！」 喜鵲小姐第一個睜開眼睛， 打開綠色...
1,就被請走給老杉樹公公去看病。 不帶體溫計， 也沒有聽診器， 他仔細地給老杉樹檢查， 用他那長...
2,就飛東飛西去訪問， 讓辛苦了一天的朋友們坐下來休息， 讓她唱幾支這世界上最好聽的歌。 狐狸和...
3,不停地對著月亮和星星講故事， 一歡喜起來就怪聲怪氣地笑。
4,【楊喚詩集】鑰匙 楊喚 我有一串鑰匙， 那拙笨短小的就像白癡和侏儒， 那姣好玲瓏的一如公主之...


In [ ]:
# ==========================================
# 步驟二：載入 CKIP 斷詞 (WS) 與詞性標記 (POS) 模型
# ==========================================
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger

print("正在載入 CKIP 模型，這可能需要幾分鐘的時間（會自動下載模型檔）...")
# 參數說明：
# model="albert-base" 是一個在效能與速度上達到很好平衡的預訓練模型。
# device=0 代表使用 Colab 的 GPU 進行運算加速。若未來在沒有 GPU 的環境執行，請改為 device=-1。
ws_driver = CkipWordSegmenter(model="albert-base", device=0)
pos_driver = CkipPosTagger(model="albert-base", device=0)
print("模型載入完成！")


正在載入 CKIP 模型，這可能需要幾分鐘的時間（會自動下載模型檔）...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/39.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/40.0M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

模型載入完成！


In [ ]:
# ==========================================
# 步驟三：實作「批次斷詞」邏輯 (Batch Processing)
# ==========================================
# 這裡會直接使用上一步產生的 df_all_segments
if not df_all_segments.empty:
    print(f"\n準備對 {len(df_all_segments)} 個文本片段進行批次處理...")

    # 將 DataFrame 中的 text_segment 欄位抽出來變成一維的 Python List
    text_list = df_all_segments['text_segment'].tolist()

    # 將整個 List 一次性餵給模型。
    # ckip-transformers 底層有很好的 batch 處理機制，一次丟整個 List 的運算效率遠高於寫 for 迴圈逐句處理。
    print("1. 正在執行斷詞 (WS)...")
    ws_results = ws_driver(text_list)

    print("2. 正在執行詞性標記 (POS)...")
    # POS 模型需要以斷詞的結果作為輸入
    pos_results = pos_driver(ws_results)

    print("批次處理完成！這兩個變數 (ws_results, pos_results) 已經儲存了所有的運算結果。")
else:
    print("錯誤：找不到資料！請確認前一步驟的 df_all_segments 變數是否存在且有內容。")


準備對 50 個文本片段進行批次處理...
1. 正在執行斷詞 (WS)...



Tokenization: 100%|██████████| 50/50 [00:00<00:00, 3593.04it/s]

Inference: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


2. 正在執行詞性標記 (POS)...


Inference: 100%|██████████| 3/3 [00:02<00:00,  1.25it/s]

批次處理完成！這兩個變數 (ws_results, pos_results) 已經儲存了所有的運算結果。


In [ ]:
# ==========================================
# 步驟四：將結果寫回並清理 DataFrame
# ==========================================
print("正在將 CKIP 運算結果整合回 DataFrame...")

# 確保運算結果的長度與原始資料表的列數一致，避免對齊錯誤
if len(ws_results) == len(df_all_segments) and len(pos_results) == len(df_all_segments):

    # 1. 寫入原始的 List 格式（方便後續程式邏輯讀取與比對）
    df_all_segments['ckip_ws'] = ws_results
    df_all_segments['ckip_pos'] = pos_results

    # 2. 新增一欄「空白鍵連接的字串格式」
    # 這是為了方便你之後寫 Regex (正規表達式) 或是肉眼檢查
    # x 是一個 list，例如 ['今天', '天氣', '好']，會被轉成 "今天 天氣 好"
    df_all_segments['ckip_ws_joined'] = df_all_segments['ckip_ws'].apply(lambda x: " ".join(x))

    # 3. 同理，把詞性也串起來方便檢查
    df_all_segments['ckip_pos_joined'] = df_all_segments['ckip_pos'].apply(lambda x: " ".join(x))

    print("資料整合成功！")
else:
    print("錯誤：CKIP 輸出結果的長度與 DataFrame 筆數不符合，請確認前面的步驟是否有中斷。")




正在將 CKIP 運算結果整合回 DataFrame...
資料整合成功！


In [ ]:
# ==========================================
# 步驟五：匯出最終版本的 CSV
# ==========================================
# 定義新的輸出檔名，避免覆蓋原本還沒斷詞的檔案
final_csv_filename = "processed_segments_with_ckip_poem.csv"

# 匯出 CSV，使用 utf-8-sig 確保在 Windows Excel 打開不會亂碼
df_all_segments.to_csv(final_csv_filename, index=False, encoding='utf-8-sig')

print(f"\n大功告成！已將帶有 CKIP 斷詞特徵的資料儲存為：{final_csv_filename}")

# 顯示前 5 筆結果預覽，檢查欄位是否都有正確長出來
display(df_all_segments[['text_segment', 'ckip_ws', 'ckip_pos', 'ckip_ws_joined']].head())


大功告成！已將帶有 CKIP 斷詞特徵的資料儲存為：processed_segments_with_ckip.csv


,text_segment,ckip_ws,ckip_pos,ckip_ws_joined
0,【楊喚詩集】童詩：森林的詩 楊喚 「太陽好！ 早晨好！」 喜鵲小姐第一個睜開眼睛， 打開綠色...,"[【, 楊喚, 詩集, 】, 童詩, ：, 森林, 的, 詩, , 楊喚 , 「, 太陽,...","[PARENTHESISCATEGORY, Nb, Na, PARENTHESISCATEG...",【 楊喚 詩集 】 童詩 ： 森林 的 詩 楊喚 「 太陽 好 ！ 早晨 好 ！ ...
1,就被請走給老杉樹公公去看病。 不帶體溫計， 也沒有聽診器， 他仔細地給老杉樹檢查， 用他那長...,"[就, 被, 請走給, 老, 杉樹, 公公, 去, 看病, 。, , 不, 帶, 體溫計,...","[D, P, VF, VH, Na, Na, D, VA, PERIODCATEGORY, ...",就 被 請走給 老 杉樹 公公 去 看病 。 不 帶 體溫計 ， 也 沒有 聽診器 ...
2,就飛東飛西去訪問， 讓辛苦了一天的朋友們坐下來休息， 讓她唱幾支這世界上最好聽的歌。 狐狸和...,"[就, 飛東飛西, 去, 訪問, ，, , 讓, 辛苦, 了, 一, 天, 的, 朋友, ...","[D, VA, D, VC, COMMACATEGORY, WHITESPACE, VL, ...",就 飛東飛西 去 訪問 ， 讓 辛苦 了 一 天 的 朋友 們 坐下來 休息 ， 讓...
3,不停地對著月亮和星星講故事， 一歡喜起來就怪聲怪氣地笑。,"[不停, 地, 對著, 月亮, 和, 星星, 講, 故事, ，, , 一, 歡喜, 起來,...","[VA, DE, P, Na, Caa, Na, VE, Na, COMMACATEGORY...",不停 地 對著 月亮 和 星星 講 故事 ， 一 歡喜 起來 就 怪聲怪氣 地 笑 。
4,【楊喚詩集】鑰匙 楊喚 我有一串鑰匙， 那拙笨短小的就像白癡和侏儒， 那姣好玲瓏的一如公主之...,"[【, 楊喚, 詩集, 】, 鑰匙, , 楊喚, , 我, 有, 一, 串, 鑰匙, ，...","[PARENTHESISCATEGORY, Nb, Na, PARENTHESISCATEG...",【 楊喚 詩集 】 鑰匙 楊喚 我 有 一 串 鑰匙 ， 那 拙笨 短小 的 就...
